# Выполнение домашнего задания 2

## 1. Загрузить позиции заказов

Нужно разобрать массив товаров в `orders.json` и загрузить его в отдельную таблицу `stg.order_items`.

Одна строка в `stg.order_items` = одна позиция внутри заказа.

In [1]:
import psycopg2
import os  
import pandas as pd
import json                                                                                   
PG_DSN = (
    f"postgresql://{os.environ.get('POSTGRES_USER', 'admin')}:"
    f"{os.environ.get('POSTGRES_PASSWORD', 'admin')}@"  # No password set in your compose
    f"{os.environ.get('POSTGRES_HOST', 'localhost')}:"
    f"{os.environ.get('POSTGRES_PORT', '5432')}/"
    f"{os.environ.get('POSTGRES_DB', 'dwh')}"
)

In [2]:
with open("../seminar_02_raw/data/orders.json", "r", encoding="utf-8") as f:
    orders_data = json.load(f)

Из описания возьмем поля order_id, customer.user_id, customer.city, created_at, payment.status, payment.amount, payment.currency, из items поля line_no, product_id, sku, quantity, unit_price, category_path  + item_amount - quantity * unit_price + category - первый элемент из category_path

In [3]:
with psycopg2.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS stg.order_items (
                order_id BIGINT NOT NULL,
                user_id INT NOT NULL,
                city TEXT,
                created_at TIMESTAMPTZ,
                payment_status TEXT,
                payment_amount DECIMAL(10, 2),
                payment_currency TEXT,
                line_no INT NOT NULL,
                product_id INT NOT NULL,
                sku TEXT,
                quantity INT NOT NULL,
                unit_price DECIMAL(10, 2),
                category_path TEXT[],
                item_amount DECIMAL(10, 2),
                category TEXT,
                PRIMARY KEY (order_id, line_no)
            )
            """
        )
        conn.commit() 

In [4]:
rows_to_insert = [
    (
        order["order_id"],
        order["customer"]["user_id"],
        order["customer"]["city"],
        order["created_at"],
        order["payment"]['status'].lower(),
        float(order["payment"]['amount']),
        order["payment"]['currency'],
        item['line_no'],
        item['product_id'],
        item['sku'],
        int(item['quantity']),
        float(item['unit_price']),
        item['category_path'],
        round(int(item['quantity']) * float(item['unit_price']),2),
        item['category_path'][0]
    )
    for order in orders_data
    for item in order['items']
]

order_items_df = pd.DataFrame(rows_to_insert, columns=[
    'order_id', 'user_id', 'city', 'created_at', 'payment_status',
    'payment_amount', 'payment_currency', 'line_no', 'product_id',
    'sku', 'quantity', 'unit_price', 'category_path', 'item_amount', 'category'])

In [5]:
with psycopg2.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.executemany(
            """
            INSERT INTO stg.order_items(
                order_id,
                user_id,
                city,
                created_at,
                payment_status,
                payment_amount,
                payment_currency,
                line_no,
                product_id,
                sku,
                quantity,
                unit_price,
                category_path,
                item_amount,
                category
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            on conflict (order_id, line_no)
            do update set
                user_id = EXCLUDED.user_id,
                city = EXCLUDED.city,
                created_at = EXCLUDED.created_at,
                payment_status = EXCLUDED.payment_status,
                payment_amount = EXCLUDED.payment_amount,
                payment_currency = EXCLUDED.payment_currency,
                product_id = EXCLUDED.product_id,
                sku = EXCLUDED.sku,
                quantity = EXCLUDED.quantity,
                unit_price = EXCLUDED.unit_price,
                category_path = EXCLUDED.category_path,
                item_amount = EXCLUDED.item_amount,
                category = EXCLUDED.category
            """,
            rows_to_insert,
        )
        conn.commit()

In [6]:
with psycopg2.connect(PG_DSN) as conn:
    orders_items_df = pd.read_sql("SELECT * FROM stg.order_items", conn)
    
orders_items_df

/var/folders/k7/hcnsll657j9bntk3g21mgtzc0000gq/T/ipykernel_44878/2903435656.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders_items_df = pd.read_sql("SELECT * FROM stg.order_items", conn)


,order_id,user_id,city,created_at,payment_status,payment_amount,payment_currency,line_no,product_id,sku,quantity,unit_price,category_path,item_amount,category
0,10001,1,Chicago,2026-05-01 10:15:00+00:00,paid,352.38,USD,1,1,ESS-001,2,129.99,"[beauty, makeup]",259.98,beauty
1,10001,1,Chicago,2026-05-01 10:15:00+00:00,paid,352.38,USD,2,2,EYE-002,1,99.90,"[beauty, makeup]",99.90,beauty
2,10002,2,Houston,2026-05-02 12:00:00+00:00,paid,79.50,USD,1,3,POW-003,1,79.50,[fragrances],79.50,fragrances


## 2. Загрузить платежи

In [7]:
orders_data[0]['payment']

{'payment_id': 'pay_10001',
 'status': 'PAID',
 'amount': '352.38',
 'currency': 'USD',
 'attempts': [{'provider': 'stripe',
   'status': 'failed',
   'attempted_at': '2026-05-01T10:15:45Z',
   'error': {'code': 'insufficient_funds'}},
  {'provider': 'stripe',
   'status': 'success',
   'attempted_at': '2026-05-01T10:16:12Z',
   'error': None}]}

Нужно разобрать этот объект и загрузить его в отдельную таблицу `stg.payments`.

Одна строка в `stg.payments` = один платёж по заказу.

In [8]:
with psycopg2.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS stg.payments (
                payment_id TEXT PRIMARY KEY,
                order_id BIGINT,
                status TEXT,
                amount DECIMAL (10,2),
                currency TEXT
            )
            """
        )
        conn.commit() 

In [9]:
payments_to_insert = [
    (
        order['payment']['payment_id'],
        order['order_id'],
        order['payment']['status'].lower(),
        order['payment']['amount'],
        order['payment']['currency']
    )
    for order in orders_data
]
payments_df = pd.DataFrame(payments_to_insert, columns=[
    'payment_id', 'order_id', 'status', 'amount', 'currency'])

In [10]:
with psycopg2.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.executemany(
            """
            INSERT INTO stg.payments(
                payment_id,
                order_id,
                status,
                amount,
                currency
            )
            VALUES (%s, %s, %s, %s, %s)
            on conflict (payment_id)
            do update set
            order_id = excluded.order_id,
            status = excluded.status,
            amount = excluded.amount,
            currency = excluded.currency
            """,
            payments_to_insert,
        )
        conn.commit()

In [11]:
with psycopg2.connect(PG_DSN) as conn:
    df_payments = pd.read_sql("Select * from stg.payments", conn)

df_payments

/var/folders/k7/hcnsll657j9bntk3g21mgtzc0000gq/T/ipykernel_44878/2565845251.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_payments = pd.read_sql("Select * from stg.payments", conn)


,payment_id,order_id,status,amount,currency
0,pay_10001,10001,paid,352.38,USD
1,pay_10002,10002,paid,79.50,USD


## Доп. задание
В `orders.json` у платежа есть массив попыток, который нужно загрузить его в таблицу

Нужно учесть, что error_code есть не у всех попыток и в некоторых записях error может быть null 

In [12]:
orders_data[0]['payment']['attempts']

[{'provider': 'stripe',
  'status': 'failed',
  'attempted_at': '2026-05-01T10:15:45Z',
  'error': {'code': 'insufficient_funds'}},
 {'provider': 'stripe',
  'status': 'success',
  'attempted_at': '2026-05-01T10:16:12Z',
  'error': None}]

In [13]:
payment_attempts_rows = [
    (
        order['payment']['payment_id'],
        attempt_no,
        attempt['provider'],
        attempt['status'].lower(),
        attempt['error']['code'] if attempt.get('error') and attempt['error'].get('code') else None,
        attempt['attempted_at']
    )
    for order in orders_data
    for attempt_no, attempt in enumerate(order['payment']['attempts'], start=1)
]

payment_attempts_df = pd.DataFrame(payment_attempts_rows, columns=[
    'payment_id', 'attempt_no', 'provider', 'status', 'error_code', 'attempted_at'])

In [14]:
df_attempts = pd.DataFrame(payment_attempts_rows)
df_attempts

,0,1,2,3,4,5
0,pay_10001,1,stripe,failed,insufficient_funds,2026-05-01T10:15:45Z
1,pay_10001,2,stripe,success,None,2026-05-01T10:16:12Z
2,pay_10002,1,paypal,success,None,2026-05-02T12:02:00Z


In [15]:
with psycopg2.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS stg.payment_attempts (
                payment_id TEXT,
                attempt_no INT,
                provider TEXT,
                status TEXT,
                error_code TEXT,
                attempted_at TIMESTAMPTZ,
                PRIMARY KEY (payment_id,attempt_no)  
            )
            """
        )

In [16]:
with psycopg2.connect(PG_DSN) as conn:
    with conn.cursor() as cur:
        cur.executemany(
            """ 
            INSERT INTO stg.payment_attempts (
                payment_id,
                attempt_no,
                provider,
                status,
                error_code,
                attempted_at
            )
            VALUES (%s, %s, %s, %s, %s, %s)
            on conflict (payment_id,attempt_no)  
            do update set 
                attempt_no = excluded.attempt_no,
                provider = excluded.provider,
                status = excluded.status,
                error_code = excluded.error_code,
                attempted_at = excluded.attempted_at
            """,
            payment_attempts_rows
        )
        conn.commit()

In [17]:
with psycopg2.connect(PG_DSN) as conn:
    attempts_df = pd.read_sql("SELECT * FROM stg.payment_attempts", conn)

attempts_df

/var/folders/k7/hcnsll657j9bntk3g21mgtzc0000gq/T/ipykernel_44878/1371646852.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  attempts_df = pd.read_sql("SELECT * FROM stg.payment_attempts", conn)


,payment_id,attempt_no,provider,status,error_code,attempted_at
0,pay_10001,1,stripe,failed,insufficient_funds,2026-05-01 10:15:45+00:00
1,pay_10001,2,stripe,success,None,2026-05-01 10:16:12+00:00
2,pay_10002,1,paypal,success,None,2026-05-02 12:02:00+00:00


## Задание со звёздочкой

Попробуем убрать дублирование кода через общую функцию, которая будет 

1. принимать DataFrame;
2. принимать имя таблицы;
3. принимать список колонок для ON CONFLICT;
4. сама брать список колонок из DataFrame;
5. сама собирать INSERT;
6. сама собирать DO UPDATE SET;
7. загружать данные через psycopg;
8. не падать, если DataFrame пустой.

In [18]:
def load_df_to_postgres(
        df, table_name, conflict_columns
):
    if df.empty:
        print('EMPTY DATAFRAME. quitting...')
        return 0
    
    else:
        df_tuples = [tuple(row) for row in df.to_numpy()]
        columns = df.columns.tolist()
        sep_columns = ', '.join(columns)
        values = ', '.join(['%s']*len(columns))

        if len(conflict_columns)>0:

            sep_conflict_cols = ', '.join(conflict_columns)
            uptade_cols = [col for col in columns if col not in conflict_columns]
            update_set = ', '.join([f"{uptade_col} = EXCLUDED.{uptade_col}" for uptade_col in  uptade_cols])

            frase = f"""
                    INSERT INTO {table_name} ({sep_columns})
                    VALUES ({values})
                    ON CONFLICT ({sep_conflict_cols})
                    DO UPDATE SET {update_set}
                    """
        else:
            frase = f"""INSERT INTO {table_name} ({sep_columns} )
                        VALUES ({values})"""


        with psycopg2.connect(PG_DSN) as conn:
            with conn.cursor() as cur:
                cur.executemany(frase,df_tuples)
                conn.commit()
        return 

In [19]:
load_df_to_postgres(order_items_df, "stg.order_items", ['order_id', 'line_no'])
load_df_to_postgres(payments_df, "stg.payments", ['payment_id'])
load_df_to_postgres(payment_attempts_df, "stg.payment_attempts", ['payment_id'," attempt_no"])

In [20]:
with psycopg2.connect(PG_DSN) as conn:
    df_orders_items = pd.read_sql("Select * from stg.order_items", conn)
    df_payments = pd.read_sql("Select * from stg.payments", conn)
    df_payments_attempts = pd.read_sql("Select * from stg.payment_attempts", conn)

display(df_orders_items)
display(df_payments)
df_payments_attempts

/var/folders/k7/hcnsll657j9bntk3g21mgtzc0000gq/T/ipykernel_44878/233912833.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_orders_items = pd.read_sql("Select * from stg.order_items", conn)
/var/folders/k7/hcnsll657j9bntk3g21mgtzc0000gq/T/ipykernel_44878/233912833.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_payments = pd.read_sql("Select * from stg.payments", conn)
/var/folders/k7/hcnsll657j9bntk3g21mgtzc0000gq/T/ipykernel_44878/233912833.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_payments_attem

,order_id,user_id,city,created_at,payment_status,payment_amount,payment_currency,line_no,product_id,sku,quantity,unit_price,category_path,item_amount,category
0,10001,1,Chicago,2026-05-01 10:15:00+00:00,paid,352.38,USD,1,1,ESS-001,2,129.99,"[beauty, makeup]",259.98,beauty
1,10001,1,Chicago,2026-05-01 10:15:00+00:00,paid,352.38,USD,2,2,EYE-002,1,99.90,"[beauty, makeup]",99.90,beauty
2,10002,2,Houston,2026-05-02 12:00:00+00:00,paid,79.50,USD,1,3,POW-003,1,79.50,[fragrances],79.50,fragrances


,payment_id,order_id,status,amount,currency
0,pay_10001,10001,paid,352.38,USD
1,pay_10002,10002,paid,79.50,USD


,payment_id,attempt_no,provider,status,error_code,attempted_at
0,pay_10001,1,stripe,failed,insufficient_funds,2026-05-01 10:15:45+00:00
1,pay_10001,2,stripe,success,None,2026-05-01 10:16:12+00:00
2,pay_10002,1,paypal,success,None,2026-05-02 12:02:00+00:00
